# Notebook 4 — Clasificadores RNN (LSTM, BiLSTM, GRU, SimpleRNN)
**Proyecto:** Ernesto Investing AI — iDeSo

Entrena las 4 arquitecturas para los 5 tickers del proyecto y guarda los resultados en MongoDB (colección `clasificaciones_rnn`), para que `Notebook3_API.ipynb` los sirva sin volver a entrenar (igual filosofía que el SVC del Notebook 2: **entrenar una vez, servir muchas**).

Consume `precios_ohlcv` (poblada por el Notebook 1) en vez de volver a descargar de yfinance, para que RNN y SVC vean exactamente los mismos datos.

## 1. Instalación e importación de librerías

In [ ]:
!pip install -q tensorflow scikit-learn pymongo[srv] certifi

import numpy as np
import pandas as pd
import json
import os
from datetime import datetime, timezone
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, GRU, SimpleRNN

np.random.seed(42)
tf.random.set_seed(42)
print("Librerías cargadas correctamente.")

## 2. Conexión a MongoDB Atlas

In [ ]:
from google.colab import userdata
import certifi
from pymongo import MongoClient

MONGO_URI = userdata.get('MONGO_URI')

cliente = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
db = cliente['ernesto_investing_ai']

col_precios = db['precios_ohlcv']       # poblada por el Notebook 1 (entrada)
col_rnns = db['clasificaciones_rnn']    # la puebla este notebook (salida)
col_rnns.create_index('ticker', unique=True)

cliente.admin.command('ping')
print('✅ Conexión a MongoDB Atlas exitosa')
print('Documentos en precios_ohlcv:', col_precios.count_documents({}))

## 3. Configuración común

Mismos 5 tickers del proyecto. Ventana de 10 días (igual que el resto de indicadores rolling usados por el SVC), 4 features por muestra.

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
VENTANA = 10
FEATURES = ["Close", "SMA_20", "EMA_12", "RSI_14"]
EPOCHS = 40
PORCENTAJE_TRAIN = 0.8


def cargar_precios(ticker: str) -> pd.DataFrame:
    """Lee precios_ohlcv desde MongoDB (poblada por el Notebook 1), ordenado por fecha."""
    cursor = col_precios.find({'ticker': ticker}).sort('fecha', 1)
    df = pd.DataFrame(list(cursor))
    if df.empty:
        return df
    df = df.rename(columns={
        'Open': 'Open', 'High': 'High', 'Low': 'Low', 'Close': 'Close', 'Volume': 'Volume',
    })
    return df

## 4. Ventanas deslizantes, entrenamiento y evaluación (4 arquitecturas × 5 tickers)

In [ ]:
def construir_ventanas(matriz: np.ndarray, target: np.ndarray, ventana: int):
    X, y = [], []
    for i in range(len(matriz) - ventana):
        X.append(matriz[i:i + ventana])
        y.append(target[i + ventana])
    return np.array(X), np.array(y)


def evaluar_modelo(y_real, y_pred_prob):
    y_pred = (y_pred_prob > 0.5).astype(int)
    return {
        "accuracy": round(float(accuracy_score(y_real, y_pred)), 4),
        "precision": round(float(precision_score(y_real, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_real, y_pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y_real, y_pred, zero_division=0)), 4),
    }


def obtener_senal(probabilidad):
    if probabilidad > 0.65: return "BUY"
    if probabilidad < 0.35: return "SELL"
    return "HOLD"


def construir_arquitectura(kind: str, input_shape):
    m = Sequential()
    capa = {
        "lstm": LSTM(64, input_shape=input_shape),
        "bilstm": Bidirectional(LSTM(64), input_shape=input_shape),
        "gru": GRU(64, input_shape=input_shape),
        "simplernn": SimpleRNN(64, input_shape=input_shape),
    }[kind]
    m.add(capa)
    m.add(Dense(16, activation='relu'))
    m.add(Dense(1, activation='sigmoid'))
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m


def entrenar_rnns_para_ticker(ticker: str) -> dict:
    df = cargar_precios(ticker)
    if df.empty or len(df) < (VENTANA + 40):
        print(f"  ⚠️ {ticker}: datos insuficientes en MongoDB, se omite.")
        return None

    df = df.dropna(subset=FEATURES).reset_index(drop=True)
    df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    df_modelo = df.iloc[:-1].dropna(subset=FEATURES + ['target']).reset_index(drop=True)

    X_raw = df_modelo[FEATURES].values
    y_raw = df_modelo['target'].values
    X_vent, y_vent = construir_ventanas(X_raw, y_raw, VENTANA)

    n = len(X_vent)
    corte = int(n * PORCENTAJE_TRAIN)
    X_train, X_test = X_vent[:corte], X_vent[corte:]
    y_train, y_test = y_vent[:corte], y_vent[corte:]

    n_feat = X_raw.shape[1]
    scaler = MinMaxScaler()
    scaler.fit(X_train.reshape(-1, n_feat))
    X_train_s = scaler.transform(X_train.reshape(-1, n_feat)).reshape(X_train.shape)
    X_test_s = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape)
    ultima_ventana_s = scaler.transform(X_raw[-VENTANA:]).reshape(1, VENTANA, n_feat)

    resultados_modelos = {}
    for kind in ["lstm", "bilstm", "gru", "simplernn"]:
        modelo = construir_arquitectura(kind, (VENTANA, n_feat))
        modelo.fit(X_train_s, y_train, epochs=EPOCHS, batch_size=32, verbose=0)
        pred_prob = modelo.predict(X_test_s, verbose=0).flatten()
        metricas = evaluar_modelo(y_test, pred_prob)
        prob_mañana = float(modelo.predict(ultima_ventana_s, verbose=0).flatten()[0])
        resultados_modelos[kind] = {
            "metricas": metricas,
            "probabilidad_mañana": prob_mañana,
            "senal": obtener_senal(prob_mañana),
        }
        print(f"  {ticker} · {kind:10s} → acc={metricas['accuracy']:.3f}  señal={resultados_modelos[kind]['senal']}")

    return {
        "ticker": ticker,
        "ultima_actualizacion": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        "modelos": resultados_modelos,
    }

## 5. Ejecutar para los 5 tickers y guardar en MongoDB (+ respaldo JSON local)

In [ ]:
os.makedirs('data', exist_ok=True)
resultados_todos = {}

for ticker in TICKERS:
    print(f"\n=== Entrenando RNNs para {ticker} ===")
    contrato = entrenar_rnns_para_ticker(ticker)
    if contrato is None:
        continue

    # Upsert en MongoDB: un documento por ticker, se sobreescribe en cada corrida
    col_rnns.update_one(
        {'ticker': ticker},
        {'$set': {**contrato, 'created_at': datetime.now(timezone.utc)}},
        upsert=True,
    )
    resultados_todos[ticker] = contrato

# Respaldo JSON local (carpeta data/, como indica la estructura del proyecto)
with open('data/clasificaciones_rnn.json', 'w') as f:
    json.dump(resultados_todos, f, indent=2, ensure_ascii=False)

print(f"\n✅ Listo. {len(resultados_todos)}/{len(TICKERS)} tickers guardados en 'clasificaciones_rnn' "
      f"y en data/clasificaciones_rnn.json")